# E2 · TruncatedSVD does not center: the first component is the mean

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Find out why a reduction to 2 dimensions with `TruncatedSVD` on the digits gives worse results than `PCA` (accuracy 0.41 against 0.60) and an "explained variance" that looks small: `TruncatedSVD` **does not center the data**, so its first component is almost the direction of the mean. Fix it by centering first.

## Dataset

`load_digits`: 1797 images, 64 pixels (0-16, all nonnegative: the mean is far from zero). Validation `StratifiedKFold(5, shuffle=True, random_state=0)`.

## Educational purpose

Recognize when two algorithms with similar names are not equivalent. `TruncatedSVD` (SVD without centering) is designed for **sparse** matrices (text, counts), where centering would destroy the sparsity. On dense data with a nonzero mean it is not a PCA.

## Task

1. Run the colleague's cell: `svd_acc` (`TruncatedSVD(2)` + logistic) and `pca_acc` (`PCA(2)` + logistic): the colleague chose the wrong one.
2. Check what the first component contains: `svd_cosine` (|cosine| between `TruncatedSVD.components_[0]` and the normalized mean vector of the data) and `pca_cosine` (the same for `PCA.components_[0]`).
3. Compare the explained variance: `svd_evr` and `pca_evr` (sum of `explained_variance_ratio_` with 2 components). Which one is physically comparable?
4. Fix it: `centered_svd_acc` = accuracy with a pipeline `StandardScaler(with_std=False)` (centering only) → `TruncatedSVD(2, algorithm="arpack")` → logistic. It must match (within 0.01) `pca_acc`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
mean_direction = X.mean(0) / np.linalg.norm(X.mean(0))

## Your work

### The colleague's code

In [ ]:
svd_acc = cross_val_score(make_pipeline(TruncatedSVD(2, random_state=0), LogisticRegression(max_iter=3000)), X, y, cv=cv).mean()
pca_acc = cross_val_score(make_pipeline(PCA(2), LogisticRegression(max_iter=3000)), X, y, cv=cv).mean()
print(f"accuracy with 2 components: TruncatedSVD {svd_acc:.3f} | PCA {pca_acc:.3f}")

### Your diagnosis and the fix

In [ ]:
svd_cosine = None
pca_cosine = None
svd_evr = None
pca_evr = None

In [ ]:
centered_svd_acc = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`svd = TruncatedSVD(2, random_state=0).fit(X); abs(svd.components_[0] @ mean_direction)`. Centered pipeline: `make_pipeline(StandardScaler(with_std=False), TruncatedSVD(2, algorithm="arpack"), LogisticRegression(max_iter=3000))`.'):
    check(svd_acc < 0.5 and pca_acc > 0.55, "TruncatedSVD below 0.5, PCA above 0.55", "don't modify the colleague's code")
    check(svd_cosine is not None and svd_cosine > 0.99 and pca_cosine < 0.05, "the first component of TruncatedSVD coincides with the direction of the mean (cosine > 0.99); the PCA one is almost orthogonal (< 0.05)", "np.abs(components_[0] @ mean_direction)")
    check(svd_evr is not None and svd_evr < 0.2 and pca_evr > 0.27, "TruncatedSVD 'explains' less than 20%, PCA over 27%: they are not comparable", "sum(explained_variance_ratio_)")
    check(centered_svd_acc is not None and abs(centered_svd_acc - pca_acc) < 0.01, "after centering the data TruncatedSVD and PCA give the same result (within 0.01)", "StandardScaler(with_std=False) before TruncatedSVD")
done()

---
The commented solution is in `solutions/E_find_the_bug/E2_truncatedsvd_instead_of_pca.ipynb`. Open it after you pass the check, or after you have really tried.